In [231]:
import torch
import torch.nn as nn
import numpy as np

In [232]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cpu'

In [233]:
data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'ratings': [1, 0, 1, 0, 1, 0] # 1: 긍정, 0: 부정
}

In [234]:
import pandas as pd
df = pd.DataFrame(data)
df.head(1)

,reviews,ratings
0,이 영화 정말 좋아 최고야,1


In [235]:
# 전처리
import re
def clean_data(text):
    return re.sub(r'[^가-힣\s]', '', text)
df['clean'] = df['reviews'].apply(lambda x : clean_data(x))

# 토큰분류 형태소
from konlpy.tag import Okt
okt = Okt()
def kor_tokened(text):    
    return [word for word, pos in okt.pos(text,stem=True) if len(word) > 1 and pos in ['Noun','Verb','Adjective']]

df['tokens'] = df['clean'].apply(lambda x : kor_tokened(x))

In [236]:
df['tokens']

0        [영화, 정말, 좋다, 최고]
1      [시간, 아깝다, 쓰레기, 영화]
2          [배우, 연기, 훌륭하다]
3        [스토리, 지루하다, 뻔하다]
4    [인생, 최고, 명작, 이다, 추천]
5       [주다, 보기, 아깝다, 졸작]
Name: tokens, dtype: object

In [237]:
# 단어인덱싱
vocab = {
    '<PAD>' : 0,
    '<UNK>' : 1
}
def make_vocab(tokens):
    for token in tokens:    
        if token not in vocab:
            vocab[token] = len(vocab)
df['tokens'].apply(lambda x : make_vocab(x))

0    None
1    None
2    None
3    None
4    None
5    None
Name: tokens, dtype: object

In [238]:
# padding  길이 맞추기 
MAX_LEN = 5
def to_sequence(tokens):
    # 토큰화 - 패딩    
    x = [vocab.get(word,1) for word in tokens ]
    if len(x) < MAX_LEN:
        x = x + [0]*(MAX_LEN-len(x))
    else:
        x = x[:MAX_LEN]
    return x

df['pad_tokens'] =  df['tokens'].apply(lambda x : to_sequence(x))
df

,reviews,ratings,clean,tokens,pad_tokens
0,이 영화 정말 좋아 최고야,1,이 영화 정말 좋아 최고야,"[영화, 정말, 좋다, 최고]","[2, 3, 4, 5, 0]"
1,시간 아까운 쓰레기 영화,0,시간 아까운 쓰레기 영화,"[시간, 아깝다, 쓰레기, 영화]","[6, 7, 8, 2, 0]"
2,배우들 연기가 너무 훌륭해요,1,배우들 연기가 너무 훌륭해요,"[배우, 연기, 훌륭하다]","[9, 10, 11, 0, 0]"
3,스토리가 지루하고 뻔하다,0,스토리가 지루하고 뻔하다,"[스토리, 지루하다, 뻔하다]","[12, 13, 14, 0, 0]"
4,인생 최고의 명작입니다 추천,1,인생 최고의 명작입니다 추천,"[인생, 최고, 명작, 이다, 추천]","[15, 5, 16, 17, 18]"
5,돈 주고 보기 아까운 졸작,0,돈 주고 보기 아까운 졸작,"[주다, 보기, 아깝다, 졸작]","[19, 20, 7, 21, 0]"


In [239]:
# 데이터셋, -- > Tensor타입변경 
# 데이터로더 --> 배치단위로 학습
from torch.utils.data  import Dataset, DataLoader
class ReviewMovieDataset(Dataset):
    def __init__(self,x,y):
        self.x = torch.LongTensor(x)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, index):
        return self.x[index], self.y[index]
x = df['pad_tokens'].tolist();  y = df['ratings'].tolist()
dataset = ReviewMovieDataset(x,y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)        

In [240]:
print( next(iter(dataloader)) )

[tensor([[ 9, 10, 11,  0,  0],
        [12, 13, 14,  0,  0]]), tensor([1., 0.])]


In [241]:
VOCAB_SIZE = len(vocab)

In [242]:
import torch.nn.functional as F
class TextCNN(nn.Module):
    def __init__(self, vocab_size,embedding_dim, filter_sizes, num_fillers):
        super().__init__()        
        self.embedding = nn.Embedding(num_embeddings = vocab_size, embedding_dim=embedding_dim)
        # 합성곱 계층  텍스트는 이미지의 흑백처럼 채널을 1로 해서 사용
        # in_channel = 1(흑백이미지처럼 채널이 1)
        # kernel_size =(n-gram크기, 임베딩차원) -> 단어가 쪼개지지 않도록 width embedding dim과 동일
        self.convs =  nn.ModuleList(
                        [nn.Conv2d(in_channels=1, out_channels = num_fillers,
                                kernel_size=(fs, embedding_dim))
                                for fs in filter_sizes]
        )
        # 완전연결층,FC,분류기 입력
        # 추출된특징들을 모두 이어 붙임
        self.fc = nn.Linear(len(filter_sizes)*num_fillers, 1) # 이진분류
        self.dropout = nn.Dropout(0.5)
    def forward(self, x):
        # x.shape (batch, seq_len)  (2, 5)
        # 임베딩 적용 -> (batch,seq_len,embedding_dim) (2,5,embedding_dim)
        x = self.embedding(x)
        # conv2d 입력(batch, channel, height, width) 채널정보를 받아야해서
        # shape : (batch,1,seq_len, embedding_dim)
        x = x.unsqueeze(1)
        pooled_outputs = []
        for conv in self.convs:
            # 합성곱, Relu 연산  shape : (batch,num_filters, seq_len-filter_size+1 ,1)
            c = F.relu(conv(x))  # shape(batch, num_filters,seq_len-filter_size+1)
            c = c.squeeze(3)
            # 최대폴링 : 문장에서 가장 강력한 특징 1개만 추출
            p = F.max_pool1d(c, c.size(2)) # shape (batch, num_filters,1)
            pooled_outputs.append(p.squeeze(2)) # shape (batch,num_filters)
        # 분류(추출된 피처들을 concatenate 후 FC레이어 통과)
        x_cat = torch.concatenate(pooled_outputs, dim=1) # shape(Batch, num_filters*len(filter_sizes))
        x_cat = self.dropout(x_cat)

        logits = self.fc(x_cat) # shape (Batch,1)
        return logits.squeeze(1) # shape (batch)

In [243]:
# 모델 셋업
VOCAB_SIZE = len(vocab)
EMBED_DIM = 16 # 단어를 16차원 벡터로 표현
FILTER_SIZES = [2,3,4,5]  # 바이어그램(2), 트라이그램(3)사용
NUM_FILTERS = 10  # 각 사이즈별 필터 개수
model = TextCNN(VOCAB_SIZE,EMBED_DIM,FILTER_SIZES,NUM_FILTERS)

In [244]:
model

TextCNN(
  (embedding): Embedding(22, 16)
  (convs): ModuleList(
    (0): Conv2d(1, 10, kernel_size=(2, 16), stride=(1, 1))
    (1): Conv2d(1, 10, kernel_size=(3, 16), stride=(1, 1))
    (2): Conv2d(1, 10, kernel_size=(4, 16), stride=(1, 1))
    (3): Conv2d(1, 10, kernel_size=(5, 16), stride=(1, 1))
  )
  (fc): Linear(in_features=40, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [245]:
from torch.optim import Adam
criterion = nn.BCEWithLogitsLoss()  # 내부에 sigmoid 포함
optimizer = Adam(model.parameters(), lr=1e-3)

In [246]:
from tqdm import tqdm
epochs = 10
model.train()
for epoch in tqdm(range(epochs)):
    total_loss = 0.0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions,batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'epoch : {epoch+1} / {epochs} loss : {( total_loss / len(dataloader) ):.4f}')

  0%|          | 0/10 [00:00<?, ?it/s]

epoch : 1 / 10 loss : 0.6415

100%|██████████| 10/10 [00:00<00:00, 46.81it/s]


epoch : 2 / 10 loss : 0.6437
epoch : 3 / 10 loss : 0.6705
epoch : 4 / 10 loss : 0.5986
epoch : 5 / 10 loss : 0.5994
epoch : 6 / 10 loss : 0.5417
epoch : 7 / 10 loss : 0.5001
epoch : 8 / 10 loss : 0.5912
epoch : 9 / 10 loss : 0.4292
epoch : 10 / 10 loss : 0.5685


100%|██████████| 10/10 [00:00<00:00, 44.23it/s]


In [248]:
# 예측
model.eval()
test_reivew = '돈 주고 보기 아까운 졸작'#'이 영화 정말 좋아 최고야'
test_reivew = [
    vocab.get(word,1)
        for word in [word for word, pos in okt.pos(test_reivew,stem=True) if len(word) > 1]
]
if len(test_reivew) < MAX_LEN:
    test_reivew = test_reivew + [0]*(MAX_LEN - len(test_reivew))
test_reivew = torch.LongTensor(test_reivew)
test_reivew = test_reivew.unsqueeze(0)
with torch.no_grad():
    logits = model(test_reivew)
    prob = torch.sigmoid(logits)[0].item()
    print( '긍정' if prob > 0.5 else '부정' , prob)  

    

부정 0.40337079763412476
